# codesearch — дообучение би-энкодера на CodeSearchNet

Как в проекте [sturdy-dev/semantic-code-search](https://github.com/sturdy-dev/semantic-code-search):
берём маленькую general-модель и дообучаем её на парах (докстринг → код) с
MultipleNegativesRankingLoss (in-batch negatives).

**Честность оценки (важно!):**
- учимся ТОЛЬКО на train-сплите, меряемся на test-сплите;
- train дедуплицируется против test по хэшу кода (сплиты CodeSearchNet пересекаются) —
  иначе модель увидит целевые функции из теста и метрики будут фиктивными;
- документ-сторона — тот же структурированный текст без докстринга, что и в нашем eval
  (train/inference parity), докстринг вырезается.

**Перед запуском:** Runtime → Change runtime type → **T4 GPU**, затем Runtime → Run all.

Время: ~40–60 минут (сбор пар + 1–2 эпохи + финальный eval).

In [ ]:
!nvidia-smi

In [ ]:
!rm -rf code-searching-engine
!git clone -b feat/embedder-comparison https://github.com/PopovDanil/code-searching-engine.git
%cd code-searching-engine
!pip install -q sentence-transformers
!pip install -q faiss-cpu tree-sitter tree-sitter-python tree-sitter-java tree-sitter-javascript tree-sitter-go tree-sitter-ruby tree-sitter-php datasets typer pyyaml

## 1. Сбор обучающих пар (train-сплит, дедуп против test)

Используем парсер и подготовку примеров самого репозитория, чтобы документ-сторона
совпадала с тем, что видит модель на инференсе.

In [ ]:
import hashlib
import re
import sys

sys.path.insert(0, "src")
from datasets import load_dataset
from tqdm.auto import tqdm

from config import CodeSearchConfig
from evaluation.evaluate import _prepare_evaluation_example

LANGUAGES = ["python", "java", "javascript", "go", "ruby", "php"]
TRAIN_PER_LANG = 20000     # верхняя граница обучающих строк на язык
TEST_DEDUP_PER_LANG = None # None = весь test-сплит для построения множества дедупа

cfg = CodeSearchConfig()  # дефолтный chunking/подготовка


def code_hash(code: str) -> str:
    """Нормализованный хэш кода (для дедупа train против test)."""
    norm = " ".join(re.findall(r"\w+", code.casefold()))
    return hashlib.sha1(norm.encode()).hexdigest()


# 1a. Множество хэшей кода из TEST — то, что нельзя показывать модели
test_hashes = set()
for lang in LANGUAGES:
    ds = load_dataset("code-search-net/code_search_net", lang, split="test")
    for ex in tqdm(ds, desc=f"test hashes {lang}"):
        code = ex.get("func_code_string") or ""
        if code.strip():
            test_hashes.add(code_hash(code))
print(f"test-хэшей для дедупа: {len(test_hashes)}")

In [ ]:
from sentence_transformers import InputExample

# 1b. Пары (докстринг -> структурированный код без докстринга) из TRAIN, без утечки
train_examples = []
skipped_leak = 0
for lang in LANGUAGES:
    ds = load_dataset("code-search-net/code_search_net", lang, split="train")
    kept = 0
    for ex in tqdm(ds, desc=f"train pairs {lang}"):
        if kept >= TRAIN_PER_LANG:
            break
        code = ex.get("func_code_string") or ""
        if code.strip() and code_hash(code) in test_hashes:
            skipped_leak += 1
            continue
        query, chunks = _prepare_evaluation_example(ex, lang, cfg)
        if not query.strip() or not chunks:
            continue
        # берём первый чанк родительской функции как позитив
        doc_text = chunks[0].to_structured_text(include_docstring=False)
        train_examples.append(InputExample(texts=[query, doc_text]))
        kept += 1
    print(f"  {lang}: {kept} пар")

print(f"\nвсего обучающих пар: {len(train_examples)}; пропущено из-за утечки в test: {skipped_leak}")

## 2. Дообучение (MultipleNegativesRankingLoss)

In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses

BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # наш контроль = база для дообучения
EPOCHS = 2
BATCH = 128
OUT_DIR = "finetuned-minilm-code"

model = SentenceTransformer(BASE_MODEL)
loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH, drop_last=True)
loss = losses.MultipleNegativesRankingLoss(model)
warmup = int(len(loader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup,
    show_progress_bar=True,
    output_path=OUT_DIR,
)
print(f"модель сохранена в {OUT_DIR}")

## 3. Оценка на том же протоколе (test-сплит, 6 языков)

Сравнение: base MiniLM (конфиг 6) vs дообученная MiniLM (конфиг 8).

In [ ]:
import os
import subprocess

env = os.environ.copy()
env["PYTHONPATH"] = "src"


def run_eval(config_path):
    proc = subprocess.run(
        [
            "python", "-m", "cli", "evaluate",
            "--languages", ",".join(LANGUAGES),
            "--max-dataset-records", "10000",
            "--max-queries", "200",
            "--config", config_path,
        ],
        env=env, capture_output=True, text=True,
    )
    print(proc.stdout[-2500:])
    if proc.returncode != 0:
        print("STDERR tail:\n", proc.stderr[-2500:])


print("=" * 60, "\nBASE MiniLM\n", "=" * 60)
run_eval("configs/embedder-6-minilm.yaml")
print("=" * 60, "\nFINE-TUNED MiniLM\n", "=" * 60)
run_eval("configs/embedder-8-finetuned-minilm.yaml")

## Что дальше

1. Сравнить base vs fine-tuned MiniLM — прирост в `experiments.md`.
2. Если хочется сильнее: дообучить не MiniLM, а код-модель (CodeRankEmbed) — потолок выше.
3. Для защиты: показать, что прирост от адаптации, а не от зубрёжки — прогнать на CoIR
   (внешний бенчмарк, не CodeSearchNet).
4. Чтобы переиспользовать модель в других ноутбуках — залить её на HF Hub
   (`model.push_to_hub(...)`) или сохранить на Google Drive.